In [2]:
# preliminary functions to implement 
# agent-ish thermal slumping
# on a simple 4x4 grid

In [3]:
# assume two fields:
# - temperature -- of each node
# - is_soil     -- nodes are either soil or air

In [4]:
def is_interior(grid, x, y):
    # determine whether node is on bottom or right boundary
    return (y >= 0 and x != grid.shape[1])

In [5]:
# function 1: diag_is_soil
# only evaluating down-right diagonal
# assesses whether down-right is soil (or air)
def downright_is_open(grid, node, x, y):
    # check for not bottom or right boundary
    if is_interior(grid, node):
        x, y = grid.x_at_node(node), grid.y_at_node(node)
        diag_node = grid.find_nearest_node([x+1, y-1]) # check this
        return grid.at_node['is_soil'][diag_node] == False
    else:
        return False # safety
        

In [6]:
def is_thawed(grid, node):
    return grid.at_node["temp"][node] >= 0        

In [7]:
def get_downright_node(grid, node, x, y):
    if is_interior(grid, node):
        diag_node = grid.find_nearest_node([x+1, y-1]) # check this
        return diag_node
    else:
        return ValueError("must be an interior node")

In [8]:
def slide_node(grid, node, diag_node):
    # switch materials from clay to air and air to clay
    grid.at_node["is_soil"][node] = 0 # set current node to air
    grid.at_node["is_soil"][diag_node] = 1 # set down diag to clay
    
    # switch the temperatures
    temp_temp = grid.at_node["temp"][node]
    grid.at_node["temp"][node] = grid.at_node["temp"][diag_node]
    grid.at_node["temp"][diag_node] = temp_temp

In [9]:
def will_slide(grid, node):
    x, y = grid.x_at_node(node), grid.y_at_node(node)
    if downright_is_open(grid, node, x, y) and is_thawed(grid,node):
        diag_node = get_downright_node(grid, node, x, y)
        slide_node(grid, node, diag_node)